# Features experiment

In [1]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

mlflow.set_experiment("feature_experiments")

/home/ksenia/Desktop/Хакатон/Data_fusion/data_fusion_env/lib/python3.12/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


<Experiment: artifact_location='mlflow-artifacts:/437407864204550308', creation_time=1774025971221, experiment_id='437407864204550308', last_update_time=1774025971221, lifecycle_stage='active', name='feature_experiments', tags={}>

In [2]:
mlflow.autolog()

In [3]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from catboost import Pool, CatBoostClassifier
from sklearn.metrics import roc_auc_score

2026/03/20 20:04:04 WARNING mlflow.utils.autologging_utils: You are using an unsupported version of sklearn. If you encounter errors during autologging, try upgrading / downgrading sklearn to a supported version, or try upgrading MLflow.
2026/03/20 20:04:05 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/20 20:04:05 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.


In [4]:
train = pl.read_parquet('data/train_main_features.parquet')
test = pl.read_parquet('data/test_main_features.parquet')

print('Тренировочные данные:', train.shape)
print('Тестовые данные:', test.shape)

Тренировочные данные: (750000, 200)
Тестовые данные: (250000, 200)


In [13]:
cat_feature_names = [
    col_name for col_name in train.columns 
    if col_name.startswith("cat_feature")
]

train = train.with_columns(
    pl.col(cat_feature_names).cast(pl.Int32)
)
train = train[:,1:]

In [14]:
target = pl.read_parquet('data/train_target.parquet')
target_columns = [col for col in target.columns if col.startswith("target")]
target = target[:, 1:]

In [16]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    train,
    target,
    test_size=0.2,
    random_state=42
)

train_pool = Pool(data = X_tr.to_pandas(), 
                  label = y_tr.to_pandas(), 
                  cat_features = cat_feature_names)

In [17]:
model = CatBoostClassifier(iterations = 10, 
                           depth = 4, 
                           learning_rate = 0.25, 
                           loss_function = 'MultiLogloss', 
                           nan_mode = 'Min', 
                           random_seed = 1234,
                           task_type='GPU',        # ← добавляем эту строчку
                           devices='0', 
                           verbose = 1)

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

with mlflow.start_run(run_name="baseline_features"):

    model.fit(train_pool)

    val_preds = model.predict(X_val)

    rmse = mean_squared_error(y_val, val_preds, squared=False)

    mlflow.log_metric("val_rmse", rmse)